In [1]:
import torch
from torch import nn
import numpy as np

In [2]:
from pathlib import Path
data_path = Path("data/")
image_path = data_path/"desert101"
train_path = image_path/"train"
test_path = image_path/"test"

In [3]:
image_path

WindowsPath('data/desert101')

In [4]:
import os
def check_data(dir_path):
    for dirpath,dirnames,filenames in os.walk(dir_path):
        print(f"# of directories: {len(dirnames)} and {len(filenames)} images in {dirpath}")

In [5]:
check_data(image_path)

# of directories: 2 and 1 images in data\desert101
# of directories: 4 and 1 images in data\desert101\test
# of directories: 0 and 20 images in data\desert101\test\baklava
# of directories: 0 and 20 images in data\desert101\test\cannoli
# of directories: 0 and 20 images in data\desert101\test\cup_cakes
# of directories: 0 and 20 images in data\desert101\test\donuts
# of directories: 4 and 1 images in data\desert101\train
# of directories: 0 and 80 images in data\desert101\train\baklava
# of directories: 0 and 80 images in data\desert101\train\cannoli
# of directories: 0 and 80 images in data\desert101\train\cup_cakes
# of directories: 0 and 80 images in data\desert101\train\donuts


In [6]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [7]:
data_transform = transforms.Compose([
    transforms.Resize(size=(64,64)),
    transforms.RandomHorizontalFlip(p= 0.4),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor()
])

In [8]:
# Eğer kendi datasetimizle çalışıyorsak yani Pytorch içinde olan bir dataset değilse ImageFolder ile çağırabiliriz.
train_data = datasets.ImageFolder(
    root= train_path,
    transform= data_transform,
    target_transform= None
)

test_data = datasets.ImageFolder(
    root= test_path,
    transform= data_transform,
    target_transform= None
)

    

In [9]:
train_data

Dataset ImageFolder
    Number of datapoints: 316
    Root location: data\desert101\train
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.4)
               TrivialAugmentWide(num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
               ToTensor()
           )

In [10]:
test_data

Dataset ImageFolder
    Number of datapoints: 77
    Root location: data\desert101\test
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.4)
               TrivialAugmentWide(num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
               ToTensor()
           )

In [11]:
class_names = train_data.classes

In [12]:
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

In [13]:
NUM_WORKERS

24

In [14]:
train_dataloader = DataLoader(
    dataset= train_data,
    batch_size= 32,
    shuffle= True,
    num_workers= NUM_WORKERS
)
test_dataloader = DataLoader(
    dataset= train_data,
    batch_size= 32,
    shuffle= True,
    num_workers= NUM_WORKERS
)


In [15]:

class Dessert101CNN(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()

        self.block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape, 
                out_channels=hidden_units,
                kernel_size=(3,3),
                stride=1,
                padding=1
        ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=hidden_units, 
                out_channels=hidden_units,
                kernel_size=(3,3),
                stride=1,
                padding=1
        ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=(2,2),
                stride=2
        )
        )

        self.block_2 = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_units, 
                out_channels=hidden_units,
                kernel_size=(3,3),
                stride=1,
                padding=1
        ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=hidden_units, 
                out_channels=hidden_units,
                kernel_size=(3,3),
                stride=1,
                padding=1
        ),
            nn.ReLU(),
            nn.MaxPool2d(
                kernel_size=(2,2),
                stride=2
        )
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                in_features=hidden_units * 16 * 16,
                out_features=output_shape
            ))

        
    def forward(self,X):
        return self.classifier(self.block_2(self.block_1(X)))
                
    

In [16]:
model_0 = Dessert101CNN(
    input_shape=3,
    hidden_units=32,
    output_shape=len(class_names)
)

In [17]:
def train_step(model: torch.nn.Module, 
               dataloader: torch.utils.data.DataLoader, 
               loss_fn: torch.nn.Module, 
               optimizer: torch.optim.Optimizer):
    # Put model in train mode
    model.train()
    
    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0
    
    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item() 

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metrics across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch 
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

In [18]:
def test_step(model: torch.nn.Module, 
              dataloader: torch.utils.data.DataLoader, 
              loss_fn: torch.nn.Module):
    # Put model in eval mode
    model.eval() 
    
    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0
    
    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
    
            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()
            
            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))
            
    # Adjust metrics to get average loss and accuracy per batch 
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [19]:
# finally create a train function that combines both of them
def train(model: torch.nn.Module, 
          train_dataloader: torch.utils.data.DataLoader, 
          test_dataloader: torch.utils.data.DataLoader, 
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5):
    
    # 2. Create empty results dictionary
    results = {"train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }
    
    # 3. Loop through training and testing steps for a number of epochs
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer)
        test_loss, test_acc = test_step(model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn)
        
        # 4. Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc*100:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc*100:.4f}"
        )

        # 5. Update results dictionary
        # Ensure all data is moved to CPU and converted to float for storage
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

    # 6. Return the filled results at the end of the epochs
    return results

In [20]:
# Set random seeds
torch.manual_seed(42) 
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 10

# Recreate an instance of model
model_0 = Dessert101CNN(input_shape=3,
                  hidden_units=32, 
                  output_shape=len(train_data.classes))

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.001)

# Train model_0 
model_0_results = train(model=model_0, 
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn, 
                        epochs=NUM_EPOCHS)

Epoch: 1 | train_loss: 1.3939 | train_acc: 22.7232 | test_loss: 1.3851 | test_acc: 24.8214
Epoch: 2 | train_loss: 1.3883 | train_acc: 23.3929 | test_loss: 1.3794 | test_acc: 25.9821
Epoch: 3 | train_loss: 1.3770 | train_acc: 28.6161 | test_loss: 1.3642 | test_acc: 34.3304
Epoch: 4 | train_loss: 1.3539 | train_acc: 31.8750 | test_loss: 1.3384 | test_acc: 34.3750
Epoch: 5 | train_loss: 1.3052 | train_acc: 37.7232 | test_loss: 1.2784 | test_acc: 42.3214
Epoch: 6 | train_loss: 1.3189 | train_acc: 34.3304 | test_loss: 1.3045 | test_acc: 36.4732
Epoch: 7 | train_loss: 1.3215 | train_acc: 37.3661 | test_loss: 1.2683 | test_acc: 40.8036
Epoch: 8 | train_loss: 1.2721 | train_acc: 38.7946 | test_loss: 1.2463 | test_acc: 45.8929
Epoch: 9 | train_loss: 1.2405 | train_acc: 43.3036 | test_loss: 1.2763 | test_acc: 41.8304
Epoch: 10 | train_loss: 1.2690 | train_acc: 40.4911 | test_loss: 1.2395 | test_acc: 41.9196
